In [17]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [1]:
from util import *
import os
import torch.nn as nn
import numpy as np

# Train and Test

In [2]:
from cnn import model, input
from cnn.trainer import BaseTrainer  # assuming the code above is saved in trainer.py
from cnn.input import GenericDataLoader
from util import init_log
import copy
import qnas_config as cfg

In [3]:
LOGGER = init_log("INFO", name=__name__)

In [ ]:
phase = 'evolution'
experiment_path = 'my_exp_config3'
config_file = 'config_files_cifar/config0.txt'

args = {
    'experiment_path': experiment_path,
    'config_file': config_file,
    'data_path': 'cifar10_data',
    'log_level': 'INFO',
    'log_level': 'INFO',
    'fitness_metric': 'best_accuracy',
    'optimizer': 'AdamW',
    'data_augmentation': 'False',
    'dataset': 'cifar10',
    'save_checkpoints_epochs': 10,
    'early_stopping': 'False',
    'en_pop_crossover': 'False',
    'network_config': 'default',
    'network_gap': 'False',
    'limit_data_value': 10000,
    'batch_size': 64,
    'max_epochs': 5,
    'epochs_to_eval': 5,
    'mixed_precision': True,
    "phase": 'evolution',
    'mo_metric_base': 'loss',
    'max_params': 114090,
    'max_inference_time': 1000,
    'task': 'classification',

    
}

config = cfg.ConfigParameters(args, phase=phase)
config.get_parameters()

fn_dict=config.fn_dict
fn_dict_tf = copy.deepcopy(fn_dict)
fn_dict

net_list = ['conv_3_1_256','no_op', 'conv_3_1_128', 'no_op','no_op','no_op',
            'conv_3_1_64','conv_3_1_64','no_op','conv_3_1_256','conv_3_1_256',
            'max_pool_2_2', 'conv_3_1_128', 'no_op','no_op','no_op','no_op','no_op',
            'max_pool_2_2', 'conv_3_1_128']

params = config.train_spec

In [5]:
data_loader = GenericDataLoader(params=params)
train_loader, val_loader = data_loader.get_loader(pin_memory_device='cuda:0')

In [6]:
test_loader= data_loader.get_loader(pin_memory_device='cuda:0', for_train=False)

In [23]:
if args['dataset'].lower() in input.available_datasets:
    dataset_info = input.available_datasets[args['dataset'].lower()]
else:
    dataset_info = load_yaml(os.path.join(args['data_path'], 'data_info.txt'))

args['num_classes'] = dataset_info['num_classes']
args['task'] = dataset_info['task']
args['device'] = 'cuda:0'
args['fn_dict'] = fn_dict
args['net_list'] = net_list

In [8]:
has_cbam_key = any(key.startswith('cbam') for key in fn_dict)

In [9]:
# Create the model    
model_net = model.NetworkGraph(num_classes=dataset_info['num_classes'], 
                                network_config=args['network_config'], 
                                network_gap=args['network_gap'])

filtered_dict = {key: item for key, item in fn_dict.items() if key in net_list}
model_net.create_functions(fn_dict=filtered_dict, net_list=net_list, cbam=has_cbam_key)

# Add the fully connected layer to the model
input_shape =  [args['batch_size']] + dataset_info['shape']
inputs = torch.randn(input_shape)
with torch.no_grad():
    _ = model_net(inputs)

args['input_shape'] = input_shape

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model_net.parameters())

In [10]:
model_path = os.path.join(args['experiment_path'], '10_1')
if not os.path.exists(model_path):
    os.makedirs(model_path)

args['model_path'] = model_path

In [11]:
trainer = BaseTrainer(model_net, criterion, optimizer, train_loader, val_loader, test_loader, args, LOGGER)
results = trainer.train()
print("Training complete. Best Accuracy:", results['best_accuracy'])

INFO: trainer: 2025-03-30 21:54:13.813 - Epoch 1/5: Train Loss: 2.3152 | Train Acc: 25.01%
INFO: trainer: 2025-03-30 21:54:14.350 - Epoch 1/5: Val Loss: 1.7311 | Val Acc: 38.60%
INFO: trainer: 2025-03-30 21:54:21.401 - Epoch 2/5: Train Loss: 1.8619 | Train Acc: 33.92%
INFO: trainer: 2025-03-30 21:54:21.891 - Epoch 2/5: Val Loss: 1.5616 | Val Acc: 42.80%
INFO: trainer: 2025-03-30 21:54:28.960 - Epoch 3/5: Train Loss: 1.7155 | Train Acc: 38.72%
INFO: trainer: 2025-03-30 21:54:29.473 - Epoch 3/5: Val Loss: 1.4012 | Val Acc: 50.30%
INFO: trainer: 2025-03-30 21:54:36.608 - Epoch 4/5: Train Loss: 1.6347 | Train Acc: 41.81%
INFO: trainer: 2025-03-30 21:54:37.103 - Epoch 4/5: Val Loss: 1.4097 | Val Acc: 50.90%
INFO: trainer: 2025-03-30 21:54:44.217 - Epoch 5/5: Train Loss: 1.5125 | Train Acc: 46.07%
INFO: trainer: 2025-03-30 21:54:44.714 - Epoch 5/5: Val Loss: 1.2994 | Val Acc: 54.70%


Training complete. Best Accuracy: 54.7


In [12]:
results

{'training_losses': [2.315245442920261,
  1.8618780805004969,
  1.7154866092734866,
  1.6346766617563036,
  1.5125157899326749],
 'training_accuracies': [25.011111111111113,
  33.922222222222224,
  38.72222222222222,
  41.81111111111111,
  46.06666666666667],
 'validation_losses': [1.7311077415943146,
  1.5616343021392822,
  1.4011566042900085,
  1.4096962213516235,
  1.29940727353096],
 'validation_accuracies': [38.6, 42.8, 50.3, 50.9, 54.7],
 'best_accuracy': 54.7,
 'best_epoch': 5,
 'training_time': 38.6807005405426,
 'cuda_inference_time': 5446.434020996094,
 'total_trainable_params': 1677834,
 'model_memory_usage': 468.4921875,
 'fitness_val_loss': 43.48946841698053,
 'scalar_multi_objective': 0.6829260338489203,
 'total_flops': 2524610560,
 'confusion_matrix': None,
 'auc_score': None,
 'acc_medmnist': None,
 'test_accuracy': None,
 'test_loss': None}

# Resnet

In [13]:
from cnn import model_resnet
from cnn.trainer import ResNetTrainer  # Our subclass
import torch.nn as nn
import torch.optim as optim

In [ ]:
# Set up your parameters, data loaders, etc.
params = {
    'device': 'cuda',
    'max_epochs': 5,
    'epochs_to_eval': 5,
    'mixed_precision': True,
    'lr_scheduler': 'multistep',
    'model_path': './experiment/resnet18',
    'input_shape': [64, 3, 32, 32],
    'num_classes': 10,
    'dataset': 'cifar10',
    'data_path': 'cifar10_data',
    'phase': 'resnet',
    'fn_dict': {},  # not used for ResNet in this case
    'net_list': [],  # not used for ResNet in this case
    'task': 'classification',
}
# Instantiate your ResNet model (you can use a dummy model here, as it will be reloaded)
model_net = model_resnet.ResNet18(in_channels=3, num_classes=10)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model_net.parameters())
# Assume train_loader, val_loader, test_loader are already defined

trainer = ResNetTrainer(model_net, criterion, optimizer, train_loader, val_loader, test_loader, params)
results = trainer.train()
print("Training complete. Best Accuracy:", results['best_accuracy'])

INFO: trainer: 2025-03-30 21:54:49.527 - Epoch 1/5: Train Loss: 2.1524 | Train Acc: 22.43%
INFO: trainer: 2025-03-30 21:54:49.973 - Epoch 1/5: Val Loss: 1.9553 | Val Acc: 30.50%
INFO: trainer: 2025-03-30 21:54:54.086 - Epoch 2/5: Train Loss: 1.8625 | Train Acc: 31.42%
INFO: trainer: 2025-03-30 21:54:54.480 - Epoch 2/5: Val Loss: 1.7002 | Val Acc: 37.50%
INFO: trainer: 2025-03-30 21:54:58.627 - Epoch 3/5: Train Loss: 1.7766 | Train Acc: 35.09%
INFO: trainer: 2025-03-30 21:54:59.055 - Epoch 3/5: Val Loss: 1.6448 | Val Acc: 42.00%
INFO: trainer: 2025-03-30 21:55:03.247 - Epoch 4/5: Train Loss: 1.6879 | Train Acc: 39.02%
INFO: trainer: 2025-03-30 21:55:03.670 - Epoch 4/5: Val Loss: 1.5535 | Val Acc: 44.90%
INFO: trainer: 2025-03-30 21:55:07.857 - Epoch 5/5: Train Loss: 1.5940 | Train Acc: 42.44%
INFO: trainer: 2025-03-30 21:55:08.230 - Epoch 5/5: Val Loss: 1.5524 | Val Acc: 44.50%


Training complete. Best Accuracy: 44.9


In [15]:
results

{'training_losses': [2.1524342397848764,
  1.8624859750270844,
  1.7765636179182265,
  1.6879337165090773,
  1.5939772956901126],
 'training_accuracies': [22.433333333333334,
  31.42222222222222,
  35.08888888888889,
  39.022222222222226,
  42.44444444444444],
 'validation_losses': [1.9553042948246002,
  1.7002417743206024,
  1.6448250114917755,
  1.5534739196300507,
  1.5523514151573181],
 'validation_accuracies': [30.5, 37.5, 42.0, 44.9, 44.5],
 'best_accuracy': 44.9,
 'best_epoch': 4,
 'training_time': 22.921282052993774,
 'cuda_inference_time': 4339.146614074707,
 'total_trainable_params': 11173962,
 'model_memory_usage': 373.67041015625,
 'fitness_val_loss': None,
 'scalar_multi_objective': None,
 'total_flops': 1110845440,
 'confusion_matrix': [[500, 99, 80, 1, 3, 0, 28, 14, 122, 153],
  [12, 778, 2, 0, 0, 0, 15, 1, 8, 184],
  [122, 56, 270, 30, 156, 34, 179, 68, 22, 63],
  [38, 80, 102, 131, 43, 90, 306, 82, 20, 108],
  [48, 34, 112, 17, 313, 15, 324, 103, 10, 24],
  [15, 50, 12

# Master train script

In [18]:
from cnn import master

In [20]:
# return val is a vector of 3 values
return_val = [0, 0, 0]

In [24]:
master.fitness("10_1", args,fn_dict, net_list, train_loader, val_loader, return_val)

INFO: trainer: 2025-03-30 22:01:05.038 - Epoch 1/5: Train Loss: 2.3201 | Train Acc: 24.34%
INFO: trainer: 2025-03-30 22:01:05.038 - Epoch 1/5: Train Loss: 2.3201 | Train Acc: 24.34%
INFO: trainer: 2025-03-30 22:01:05.570 - Epoch 1/5: Val Loss: 1.8486 | Val Acc: 37.00%
INFO: trainer: 2025-03-30 22:01:05.570 - Epoch 1/5: Val Loss: 1.8486 | Val Acc: 37.00%
INFO: trainer: 2025-03-30 22:01:12.632 - Epoch 2/5: Train Loss: 1.8800 | Train Acc: 33.46%
INFO: trainer: 2025-03-30 22:01:12.632 - Epoch 2/5: Train Loss: 1.8800 | Train Acc: 33.46%
INFO: trainer: 2025-03-30 22:01:13.170 - Epoch 2/5: Val Loss: 1.6345 | Val Acc: 38.20%
INFO: trainer: 2025-03-30 22:01:13.170 - Epoch 2/5: Val Loss: 1.6345 | Val Acc: 38.20%
INFO: trainer: 2025-03-30 22:01:20.232 - Epoch 3/5: Train Loss: 1.7776 | Train Acc: 37.59%
INFO: trainer: 2025-03-30 22:01:20.232 - Epoch 3/5: Train Loss: 1.7776 | Train Acc: 37.59%
INFO: trainer: 2025-03-30 22:01:20.753 - Epoch 3/5: Val Loss: 1.4277 | Val Acc: 48.10%
INFO: trainer: 2025

{'training_losses': [2.320104443364673,
  1.8800325095653534,
  1.7776496079232957,
  1.6419856349627178,
  1.5515155924691095],
 'training_accuracies': [24.344444444444445,
  33.455555555555556,
  37.58888888888889,
  41.78888888888889,
  44.71111111111111],
 'validation_losses': [1.848573386669159,
  1.6345167458057404,
  1.42771577835083,
  1.426421344280243,
  1.3028463125228882],
 'validation_accuracies': [37.0, 38.2, 48.1, 49.8, 55.3],
 'best_accuracy': 55.3,
 'best_epoch': 5,
 'training_time': 38.121344804763794,
 'cuda_inference_time': 5456.972122192383,
 'total_trainable_params': 1677834,
 'model_memory_usage': 689.85791015625,
 'fitness_val_loss': 43.42452184333777,
 'scalar_multi_objective': 0.6890837135605793,
 'total_flops': 2524610560,
 'confusion_matrix': None,
 'auc_score': None,
 'acc_medmnist': None,
 'test_accuracy': None,
 'test_loss': None}

In [25]:
args['phase'] = 'retrain'

In [39]:
master.retrain(args,fn_dict, net_list, train_loader, val_loader, test_loader)

INFO: trainer: 2025-03-30 22:27:42.525 - Epoch 1/5: Train Loss: 2.2250 | Train Acc: 24.33%
INFO: trainer: 2025-03-30 22:27:42.525 - Epoch 1/5: Train Loss: 2.2250 | Train Acc: 24.33%
INFO: trainer: 2025-03-30 22:27:42.525 - Epoch 1/5: Train Loss: 2.2250 | Train Acc: 24.33%
INFO: trainer: 2025-03-30 22:27:42.525 - Epoch 1/5: Train Loss: 2.2250 | Train Acc: 24.33%
INFO: trainer: 2025-03-30 22:27:42.525 - Epoch 1/5: Train Loss: 2.2250 | Train Acc: 24.33%
INFO: trainer: 2025-03-30 22:27:43.063 - Epoch 1/5: Val Loss: 1.8501 | Val Acc: 37.10%
INFO: trainer: 2025-03-30 22:27:43.063 - Epoch 1/5: Val Loss: 1.8501 | Val Acc: 37.10%
INFO: trainer: 2025-03-30 22:27:43.063 - Epoch 1/5: Val Loss: 1.8501 | Val Acc: 37.10%
INFO: trainer: 2025-03-30 22:27:43.063 - Epoch 1/5: Val Loss: 1.8501 | Val Acc: 37.10%
INFO: trainer: 2025-03-30 22:27:43.063 - Epoch 1/5: Val Loss: 1.8501 | Val Acc: 37.10%
INFO: trainer: 2025-03-30 22:27:50.053 - Epoch 2/5: Train Loss: 1.8519 | Train Acc: 34.67%
INFO: trainer: 2025

{'training_losses': [2.2249698771370783,
  1.8519461353619893,
  1.7499993443489075,
  1.5939399401346843,
  1.5228572653399572],
 'training_accuracies': [24.333333333333332,
  34.666666666666664,
  38.62222222222222,
  42.9,
  45.544444444444444],
 'validation_losses': [1.8500831723213196,
  1.766331136226654,
  1.395600974559784,
  1.4347605407238007,
  1.2385696470737457],
 'validation_accuracies': [37.1, 40.5, 49.7, 49.1, 57.2],
 'best_accuracy': 57.2,
 'best_epoch': 5,
 'training_time': 37.890779972076416,
 'cuda_inference_time': 5459.499359130859,
 'total_trainable_params': 1677834,
 'model_memory_usage': 696.38525390625,
 'fitness_val_loss': None,
 'scalar_multi_objective': None,
 'total_flops': 2524610560,
 'confusion_matrix': [[637, 82, 39, 16, 36, 27, 7, 20, 97, 39],
  [24, 868, 2, 4, 9, 10, 5, 4, 7, 67],
  [107, 24, 224, 58, 286, 191, 45, 37, 17, 11],
  [23, 47, 40, 247, 133, 353, 56, 55, 15, 31],
  [39, 18, 24, 26, 653, 101, 20, 102, 12, 5],
  [8, 20, 29, 82, 87, 683, 19, 5

In [37]:
# Set up your parameters, data loaders, etc.
params = {
    'device': 'cuda',
    'max_epochs': 5,
    'epochs_to_eval': 5,
    'batch_size': 64,
    'optimizer': 'AdamW',
    'mixed_precision': True,
    'lr_scheduler': 'multistep',
    'experiment_path': './experiment/resnet18',
    'input_shape': [64, 3, 32, 32],
    'num_classes': 10,
    'dataset': 'cifar10',
    'data_path': 'cifar10_data',
    'phase': 'resnet',
    'fn_dict': {},  # not used for ResNet in this case
    'net_list': [],  # not used for ResNet in this case
    'task': 'classification',
    'model_flag': 'resnet18',
}

In [38]:
master.resnet_train(params, train_loader, val_loader, test_loader)

Using model_flag: resnet18


INFO: trainer: 2025-03-30 22:16:27.024 - Epoch 1/5: Train Loss: 2.1152 | Train Acc: 24.12%
INFO: trainer: 2025-03-30 22:16:27.024 - Epoch 1/5: Train Loss: 2.1152 | Train Acc: 24.12%
INFO: trainer: 2025-03-30 22:16:27.024 - Epoch 1/5: Train Loss: 2.1152 | Train Acc: 24.12%
INFO: trainer: 2025-03-30 22:16:27.024 - Epoch 1/5: Train Loss: 2.1152 | Train Acc: 24.12%
INFO: trainer: 2025-03-30 22:16:27.440 - Epoch 1/5: Val Loss: 1.7214 | Val Acc: 38.10%
INFO: trainer: 2025-03-30 22:16:27.440 - Epoch 1/5: Val Loss: 1.7214 | Val Acc: 38.10%
INFO: trainer: 2025-03-30 22:16:27.440 - Epoch 1/5: Val Loss: 1.7214 | Val Acc: 38.10%
INFO: trainer: 2025-03-30 22:16:27.440 - Epoch 1/5: Val Loss: 1.7214 | Val Acc: 38.10%
INFO: trainer: 2025-03-30 22:16:31.399 - Epoch 2/5: Train Loss: 1.8538 | Train Acc: 33.17%
INFO: trainer: 2025-03-30 22:16:31.399 - Epoch 2/5: Train Loss: 1.8538 | Train Acc: 33.17%
INFO: trainer: 2025-03-30 22:16:31.399 - Epoch 2/5: Train Loss: 1.8538 | Train Acc: 33.17%
INFO: trainer: 

{'training_losses': [2.115160451995002,
  1.8537752628326416,
  1.7369341916508145,
  1.6668702165285747,
  1.5625363224082522],
 'training_accuracies': [24.122222222222224,
  33.166666666666664,
  37.077777777777776,
  40.233333333333334,
  44.0],
 'validation_losses': [1.721373200416565,
  1.6666490733623505,
  1.4726928174495697,
  1.377131462097168,
  1.7421739101409912],
 'validation_accuracies': [38.1, 37.5, 44.2, 48.6, 39.7],
 'best_accuracy': 48.6,
 'best_epoch': 4,
 'training_time': 21.758289098739624,
 'cuda_inference_time': 4588.174819946289,
 'total_trainable_params': 11173962,
 'model_memory_usage': 592.4873046875,
 'fitness_val_loss': None,
 'scalar_multi_objective': None,
 'total_flops': 1110845440,
 'confusion_matrix': [[670, 18, 44, 10, 9, 39, 6, 27, 144, 33],
  [86, 569, 10, 3, 2, 16, 10, 12, 63, 229],
  [136, 7, 197, 25, 173, 249, 86, 59, 55, 13],
  [32, 10, 54, 138, 43, 505, 100, 36, 46, 36],
  [63, 5, 82, 19, 310, 214, 125, 147, 29, 6],
  [20, 4, 34, 56, 42, 720, 2